In [4]:
"""
Top 500 Crypto — 2 Years Daily OHLCV
Source : Binance Public API (free, no API key)
Coin list: CoinGecko /coins/markets (public)
Incremental: only fetches NEW dates for existing coins + NEW coins
"""

import requests
import pandas as pd
import time
import os
import json
from datetime import datetime, timedelta, timezone
from typing import Optional, List, Dict

BINANCE_URL = "https://api.binance.com/api/v3/klines"
CG_URL      = "https://api.coingecko.com/api/v3"
OUTPUT_DIR  = "crypto_data"
MASTER_FILE = "crypto_top500_2yr.csv"
CHECKPOINT  = ".checkpoint.json"

QUOTE_CURRENCIES = ["USDT", "BUSD", "BTC", "ETH", "BNB"]
STABLECOINS = {"USDT", "USDC", "BUSD", "DAI", "TUSD", "USDP", "GUSD", "FRAX",
               "LUSD", "USDD", "FDUSD", "PYUSD", "USDE", "USDX", "SUSD"}


# ── helpers ───────────────────────────────────────────────────────────────────

def now_utc() -> datetime:
    return datetime.now(timezone.utc)


def get(url: str, params: dict = {}, max_retries: int = 5, base_delay: float = 2.0):
    for attempt in range(max_retries):
        try:
            r = requests.get(url, params=params, timeout=30)
            if r.status_code == 200:
                return r.json()
            elif r.status_code == 429:
                wait = base_delay * (2 ** attempt)
                print(f"  ⏳ Rate limited — waiting {wait:.0f}s (attempt {attempt+1})")
                time.sleep(wait)
            elif r.status_code >= 500:
                wait = base_delay * (2 ** attempt)
                print(f"  ⚠ Server {r.status_code} — waiting {wait:.0f}s")
                time.sleep(wait)
            else:
                return None
        except Exception as e:
            time.sleep(base_delay * (2 ** attempt))
    return None


def load_checkpoint() -> dict:
    """
    Checkpoint stores:
      {
        "done": ["bitcoin", "ethereum", ...],        # fully fetched (initial run)
        "last_date": {"bitcoin": "2026-03-01", ...}  # last date in each coin's file
        "pair": {"bitcoin": "BTCUSDT", ...}          # binance pair used per coin
      }
    """
    if os.path.exists(CHECKPOINT):
        try:
            data = json.load(open(CHECKPOINT))
            # migrate old format (list) to new format (dict)
            if isinstance(data, list):
                return {"done": data, "last_date": {}, "pair": {}}
            return data
        except:
            pass
    return {"done": [], "last_date": {}, "pair": {}}


def save_checkpoint(cp: dict):
    json.dump(cp, open(CHECKPOINT, "w"), indent=2)


# ── step 1: top 500 from CoinGecko ───────────────────────────────────────────

def fetch_top500() -> pd.DataFrame:
    print("[1/3] Fetching top 500 list from CoinGecko...")
    coins = []
    for page in [1, 2]:
        data = get(f"{CG_URL}/coins/markets", params={
            "vs_currency": "usd",
            "order": "market_cap_desc",
            "per_page": 250,
            "page": page,
            "sparkline": "false",
        })
        if data:
            coins.extend(data)
        time.sleep(2)

    rows = []
    for c in coins:
        sym = c.get("symbol", "").upper()
        rows.append({
            "coin_id":            c.get("id"),
            "symbol":             sym,
            "name":               c.get("name"),
            "market_cap_usd":     c.get("market_cap"),
            "price_usd":          c.get("current_price"),
            "volume_24h_usd":     c.get("total_volume"),
            "change_24h_pct":     c.get("price_change_percentage_24h"),
            "high_24h":           c.get("high_24h"),
            "low_24h":            c.get("low_24h"),
            "ath":                c.get("ath"),
            "atl":                c.get("atl"),
            "circulating_supply": c.get("circulating_supply"),
            "max_supply":         c.get("max_supply"),
            "is_stablecoin":      sym in STABLECOINS,
        })

    df = pd.DataFrame(rows)

    def cap_tier(mc):
        if pd.isna(mc):  return "unknown"
        if mc >= 10e9:   return "large"
        if mc >= 1e9:    return "mid"
        if mc >= 100e6:  return "small"
        return                  "micro"

    df["cap_tier"] = df["market_cap_usd"].apply(cap_tier)
    print(f"  → {len(df)} coins loaded  ({df['is_stablecoin'].sum()} stablecoins)")
    return df


# ── step 2: Binance OHLCV ────────────────────────────────────────────────────

def find_binance_pair(symbol: str, is_stablecoin: bool, known_pair: str = None) -> Optional[str]:
    if known_pair:
        return known_pair  # reuse cached pair — no extra API call
    quotes = QUOTE_CURRENCIES if not is_stablecoin else ["USDT", "BTC"]
    for quote in quotes:
        if symbol == quote:
            continue
        pair = f"{symbol}{quote}"
        r = requests.get(BINANCE_URL, params={"symbol": pair, "interval": "1d", "limit": 1}, timeout=10)
        if r.status_code == 200 and r.json():
            return pair
    return None


def fetch_binance_ohlcv(pair: str, start_ms: int, end_ms: int) -> Optional[pd.DataFrame]:
    data = get(BINANCE_URL, params={
        "symbol":    pair,
        "interval":  "1d",
        "startTime": start_ms,
        "endTime":   end_ms,
        "limit":     1000,
    })
    if not data:
        return None

    df = pd.DataFrame(data, columns=[
        "open_time","open","high","low","close","volume",
        "close_time","quote_volume","trades",
        "taker_buy_base","taker_buy_quote","ignore"
    ])
    df["date"]       = pd.to_datetime(df["open_time"], unit="ms").dt.normalize()
    df["open"]       = df["open"].astype(float)
    df["high"]       = df["high"].astype(float)
    df["low"]        = df["low"].astype(float)
    df["close"]      = df["close"].astype(float)
    df["volume_usd"] = df["quote_volume"].astype(float)
    df["pair"]       = pair

    return df[["date","open","high","low","close","volume_usd","pair"]].sort_values("date").reset_index(drop=True)


# ── step 3: indicators ───────────────────────────────────────────────────────

def add_derived(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy().sort_values("date").reset_index(drop=True)

    df["daily_return_pct"] = (df["close"].pct_change() * 100).round(4)
    df["candle_range_pct"] = ((df["high"] - df["low"]) / df["open"] * 100).round(3)

    df["ma_20"]  = df["close"].rolling(20).mean().round(8)
    df["ma_50"]  = df["close"].rolling(50).mean().round(8)
    df["ma_200"] = df["close"].rolling(200).mean().round(8)

    df["vol_ma_20"]  = df["volume_usd"].rolling(20).mean().round(2)
    df["rel_volume"] = (df["volume_usd"] / df["vol_ma_20"]).round(3)

    # RSI-14
    delta = df["close"].diff()
    gain  = delta.clip(lower=0).rolling(14).mean()
    loss  = (-delta.clip(upper=0)).rolling(14).mean()
    rs    = gain / loss.replace(0, float("nan"))
    df["rsi_14"] = (100 - 100 / (1 + rs)).round(2)

    # Bollinger Bands (20, 2)
    std = df["close"].rolling(20).std()
    df["bb_upper"] = (df["ma_20"] + 2 * std).round(8)
    df["bb_lower"] = (df["ma_20"] - 2 * std).round(8)
    df["bb_pct"]   = ((df["close"] - df["bb_lower"]) / (df["bb_upper"] - df["bb_lower"])).round(4)

    # MACD (12, 26, 9)
    ema12             = df["close"].ewm(span=12, adjust=False).mean()
    ema26             = df["close"].ewm(span=26, adjust=False).mean()
    df["macd"]        = (ema12 - ema26).round(8)
    df["macd_signal"] = df["macd"].ewm(span=9, adjust=False).mean().round(8)
    df["macd_hist"]   = (df["macd"] - df["macd_signal"]).round(8)

    return df


# ── main ──────────────────────────────────────────────────────────────────────

def main():
    print("=" * 60)
    print("  Top 500 Crypto — Incremental OHLCV  |  Binance")
    print("=" * 60)

    os.makedirs(OUTPUT_DIR, exist_ok=True)
    cp    = load_checkpoint()
    done  = set(cp.get("done", []))
    last_date_map = cp.get("last_date", {})   # coin_id -> "YYYY-MM-DD"
    pair_map      = cp.get("pair", {})        # coin_id -> "BTCUSDT"

    meta  = fetch_top500()
    total = len(meta)

    # Classify each coin
    new_coins      = [r for _, r in meta.iterrows() if r["coin_id"] not in done]
    existing_coins = [r for _, r in meta.iterrows() if r["coin_id"] in done]

    print(f"\n[2/3] Update plan:")
    print(f"  → New coins to fetch (full 2yr)  : {len(new_coins)}")
    print(f"  → Existing coins to update       : {len(existing_coins)}")

    end_ms   = int(now_utc().timestamp() * 1000)
    all_frames = []
    start_time = time.time()
    processed  = 0

    # ── A: Update existing coins (only new dates) ─────────────────────────────
    print(f"\n  [A] Updating existing coins with new dates...")
    for row in existing_coins:
        coin_id    = row["coin_id"]
        symbol     = row["symbol"]
        is_stable  = row["is_stablecoin"]
        fpath      = os.path.join(OUTPUT_DIR, f"{coin_id}.csv")

        # Load existing data
        if not os.path.exists(fpath):
            existing_df = pd.DataFrame()
            start_ms = int((now_utc() - timedelta(days=730)).timestamp() * 1000)
        else:
            existing_df = pd.read_csv(fpath, parse_dates=["date"])
            last_dt  = existing_df["date"].max()
            # Start from day AFTER last known date
            start_ms = int((last_dt + timedelta(days=1)).timestamp() * 1000)
            last_str = last_dt.strftime("%Y-%m-%d")

        # Skip if already up to date (last date is today or yesterday)
        yesterday_ms = int((now_utc() - timedelta(days=1)).timestamp() * 1000)
        if start_ms >= yesterday_ms:
            all_frames.append(existing_df)
            processed += 1
            continue

        known_pair = pair_map.get(coin_id)
        pair = find_binance_pair(symbol, is_stable, known_pair)
        if not pair:
            all_frames.append(existing_df)
            processed += 1
            continue

        new_data = fetch_binance_ohlcv(pair, start_ms, end_ms)

        if new_data is not None and len(new_data) > 0:
            # Remove today's incomplete candle (keep only closed candles)
            new_data = new_data[new_data["date"] < pd.Timestamp(now_utc().date())]

            if len(new_data) > 0:
                # Add metadata
                for col in ["coin_id","symbol","name","market_cap_usd","cap_tier",
                            "ath","atl","circulating_supply","max_supply",
                            "change_24h_pct","volume_24h_usd","is_stablecoin"]:
                    new_data[col] = row[col]

                # Combine old + new, drop dupes, recalc indicators
                combined = pd.concat([existing_df, new_data], ignore_index=True)
                combined = combined.drop_duplicates("date").sort_values("date").reset_index(drop=True)
                combined = add_derived(combined)

                combined.to_csv(fpath, index=False)
                all_frames.append(combined)

                new_rows = len(new_data)
                last_date_map[coin_id] = combined["date"].max().strftime("%Y-%m-%d")
                print(f"  ↑ {symbol:12s} | +{new_rows} new rows | now {len(combined)} total")
            else:
                all_frames.append(existing_df)
        else:
            all_frames.append(existing_df)

        # Update checkpoint
        pair_map[coin_id] = pair
        cp = {"done": list(done), "last_date": last_date_map, "pair": pair_map}
        save_checkpoint(cp)

        processed += 1
        time.sleep(0.12)

    # ── B: Fetch new coins (full 2yr history) ─────────────────────────────────
    print(f"\n  [B] Fetching new coins (full 2yr history)...")
    start_ms_2yr = int((now_utc() - timedelta(days=730)).timestamp() * 1000)

    for idx, row in enumerate(new_coins):
        coin_id   = row["coin_id"]
        symbol    = row["symbol"]
        is_stable = row["is_stablecoin"]
        rank      = meta[meta["coin_id"] == coin_id].index[0] + 1

        pair = find_binance_pair(symbol, is_stable)
        if not pair:
            print(f"  ✗ [{rank}/{total}] {symbol:12s} — not on Binance, skipping")
            done.add(coin_id)
            cp = {"done": list(done), "last_date": last_date_map, "pair": pair_map}
            save_checkpoint(cp)
            continue

        ohlcv = fetch_binance_ohlcv(pair, start_ms_2yr, end_ms)
        if ohlcv is None or len(ohlcv) < 10:
            print(f"  ✗ [{rank}/{total}] {symbol:12s} — no data, skipping")
            done.add(coin_id)
            cp = {"done": list(done), "last_date": last_date_map, "pair": pair_map}
            save_checkpoint(cp)
            continue

        # Remove today's incomplete candle
        ohlcv = ohlcv[ohlcv["date"] < pd.Timestamp(now_utc().date())]

        for col in ["coin_id","symbol","name","market_cap_usd","cap_tier",
                    "ath","atl","circulating_supply","max_supply",
                    "change_24h_pct","volume_24h_usd","is_stablecoin"]:
            ohlcv[col] = row[col]

        ohlcv = add_derived(ohlcv)

        fpath = os.path.join(OUTPUT_DIR, f"{coin_id}.csv")
        ohlcv.to_csv(fpath, index=False)
        all_frames.append(ohlcv)

        done.add(coin_id)
        pair_map[coin_id]      = pair
        last_date_map[coin_id] = ohlcv["date"].max().strftime("%Y-%m-%d")
        cp = {"done": list(done), "last_date": last_date_map, "pair": pair_map}
        save_checkpoint(cp)

        elapsed  = time.time() - start_time
        rate     = (processed + idx + 1) / elapsed if elapsed else 1
        eta_min  = int((total - processed - idx - 1) / rate / 60) if rate else "?"
        print(f"  ✓ [{rank}/{total}] {symbol:12s} | {len(ohlcv):3d} days | {pair} | ETA ~{eta_min} min")

        time.sleep(0.12)

    # ── Merge master ──────────────────────────────────────────────────────────
    print(f"\n[3/3] Merging into master CSV...")
    if all_frames:
        master = pd.concat(all_frames, ignore_index=True)
        master["date"] = pd.to_datetime(master["date"])
        master = master.sort_values(["symbol", "date"]).reset_index(drop=True)
        master.to_csv(MASTER_FILE, index=False)

        print(f"\n{'='*60}")
        print("SUMMARY")
        print(f"{'='*60}")
        print(f"Coins in dataset : {master['symbol'].nunique()}/{total}")
        print(f"Total rows       : {len(master):,}")
        print(f"Date range       : {master['date'].min().date()} → {master['date'].max().date()}")
        print(f"Cap breakdown    : {master.drop_duplicates('coin_id')['cap_tier'].value_counts().to_dict()}")
        print(f"Columns          : {list(master.columns)}")
        print(f"\nPer-coin files   : {OUTPUT_DIR}/")
        print(f"Master CSV       : {MASTER_FILE}")

    print("\n✅ Done! Run again tomorrow — only new rows will be fetched.")


if __name__ == "__main__":
    main()

  Top 500 Crypto — Incremental OHLCV  |  Binance
[1/3] Fetching top 500 list from CoinGecko...
  → 500 coins loaded  (14 stablecoins)

[2/3] Update plan:
  → New coins to fetch (full 2yr)  : 0
  → Existing coins to update       : 500

  [A] Updating existing coins with new dates...
  ↑ BTC          | +729 new rows | now 729 total
  ↑ ETH          | +729 new rows | now 729 total
  ↑ CFG          | +4 new rows | now 4 total

  [B] Fetching new coins (full 2yr history)...

[3/3] Merging into master CSV...

SUMMARY
Coins in dataset : 213/500
Total rows       : 123,310
Date range       : 2024-03-21 → 2026-03-20
Cap breakdown    : {'small': 93, 'micro': 81, 'mid': 31, 'large': 9}
Columns          : ['date', 'open', 'high', 'low', 'close', 'volume_usd', 'pair', 'coin_id', 'symbol', 'name', 'market_cap_usd', 'cap_tier', 'ath', 'atl', 'circulating_supply', 'max_supply', 'change_24h_pct', 'volume_24h_usd', 'is_stablecoin', 'daily_return_pct', 'candle_range_pct', 'ma_20', 'ma_50', 'ma_200', 'vol_